In [ ]:
import os
import json
import shutil
import torch
import numpy as np
import pyNN.spiNNaker as sim

DATASET = "dvs_gesture"
SIMULATION_TIME_MS = 20.0
DT = 1.0
TIME_SCALE_FACTOR = 4000
WEIGHT_SCALE = 1.0
MAX_SPIKES_PER_NEURON = 10
TARGET_CONNECTIONS_PER_NEURON = 5.0

WEIGHTS_PATHS = {
    "nmnist": "networks/nmnist_best.pth",
    "cifar10_dvs": "networks/cifar10_dvs_best.pth",
    "dvs_gesture": "networks/dvs_gesture_best.pth",
    "nepic_kitchens": "networks/nepic_kitchens_best.pth",
}

TOPOLOGIES = {
    "nmnist": [2312, 256, 10],
    "cifar10_dvs": [1568, 50176, 25088, 12544, 4608, 10],
    "dvs_gesture": [32768, 131072, 65536, 32768, 16384, 11],
    "nepic_kitchens": [116736, 262144, 131072, 65536, 32768, 16384, 8192, 4096, 8]
}

LIF_PARAMETERS = {
    "v_rest": 0.0, "v_reset": 0.0, "v_thresh": 1.0,
    "tau_m": 1.4427, "tau_syn_E": 0.1, "tau_syn_I": 0.1, "tau_refrac": 0.0
}

def sort_reports(dataset_name):
    """Range les rapports de SpiNNaker dans le bon dossier."""
    report_dir = "reports"
    if not os.path.exists(report_dir): return
    subdirs = [os.path.join(report_dir, d) for d in os.listdir(report_dir) if os.path.isdir(os.path.join(report_dir, d)) and d.startswith("20")]
    if not subdirs: return
    latest_report = max(subdirs, key=os.path.getmtime)
    target_path = os.path.join(report_dir, "inference", dataset_name, os.path.basename(latest_report))
    os.makedirs(os.path.dirname(target_path), exist_ok=True)
    if not os.path.exists(target_path):
        shutil.copytree(latest_report, target_path)
        shutil.rmtree(latest_report)

def get_safe_spikes(file_path, input_size):
    """Charge les spikes et les tronque pour éviter de saturer la RAM des puces."""
    with open(file_path, 'r') as f:
        data = json.load(f)
    
    raw_spikes = data["spike_times"]
    safe_spikes = []
    
    for i in range(input_size):
        if i < len(raw_spikes) and isinstance(raw_spikes[i], list):
            safe_spikes.append(raw_spikes[i][:MAX_SPIKES_PER_NEURON])
        else:
            safe_spikes.append([])
            
    return safe_spikes, data["label"]

if __name__ == "__main__":
    pth_file = WEIGHTS_PATHS.get(DATASET, "")
    json_file = f"networks/sample_{DATASET}.json"
    
    if not os.path.exists(pth_file): raise FileNotFoundError(f"Fichier introuvable: {pth_file}")
    if not os.path.exists(json_file): raise FileNotFoundError(f"Fichier introuvable: {json_file}")

    layers_sizes = TOPOLOGIES[DATASET]
    
    print("[*] Chargement et troncature des Spikes...")
    spike_times, expected_label = get_safe_spikes(json_file, layers_sizes[0])
    
    state_dict = torch.load(pth_file, map_location="cpu", weights_only=True)
    if "state_dict" in state_dict: state_dict = state_dict["state_dict"]
    weight_keys = [k for k in state_dict.keys() if "weight" in k and len(state_dict[k].shape) >= 2]

    print("[*] Configuration de la machine SpiNNaker...")
    sim.setup(timestep=DT, time_scale_factor=TIME_SCALE_FACTOR)
    sim.set_number_of_neurons_per_core(sim.SpikeSourceArray, 16)
    sim.set_number_of_neurons_per_core(sim.IF_curr_exp, 128)

    populations = [sim.Population(layers_sizes[0], sim.SpikeSourceArray(spike_times=spike_times), label="InputLayer")]
    
    for idx, size in enumerate(layers_sizes[1:]):
        is_output = (idx == len(layers_sizes[1:]) - 1)
        params = LIF_PARAMETERS.copy()
        if is_output: params["v_thresh"] = 10000.0
        
        pop = sim.Population(size, sim.IF_curr_exp(**params), label="OutputLayer" if is_output else f"HiddenLayer_{idx}")
        if is_output: pop.record(["v"])
        populations.append(pop)

    print("[*] Création du graphe de connexions probabilistes...")
    for idx in range(len(populations) - 1):
        if idx < len(weight_keys):
            w_tensor = state_dict[weight_keys[idx]].cpu()
            
            mean_weight = float(torch.mean(torch.abs(w_tensor)).item()) * WEIGHT_SCALE
            safe_p = min(1.0, TARGET_CONNECTIONS_PER_NEURON / float(populations[idx].size))
            
            sim.Projection(
                populations[idx], 
                populations[idx+1], 
                sim.FixedProbabilityConnector(p_connect=safe_p), 
                synapse_type=sim.StaticSynapse(weight=mean_weight, delay=1.0)
            )
            
            populations[idx+1].set(i_offset=0.01)
        else:
            sim.Projection(populations[idx], populations[idx+1], sim.OneToOneConnector(weight=WEIGHT_SCALE))

    print("[*] Lancement de l'exécution matérielle (Sim: 20ms)...")
    sim.run(SIMULATION_TIME_MS)
    
    v_data = populations[-1].get_data("v").segments[0].filter(name="v")[0].magnitude
    sim.end()

    results = np.sum(v_data, axis=0)
    print(f"\nExpected class (Ground Truth) for {DATASET} : {expected_label}")
    print(f"Predicted class (SpiNNaker) : {int(np.argmax(results))}")
    print("\nVoltages accumulés par classe :\n", results)
    
    sort_reports(DATASET)